# Web Retrieval Pipeline: Step-By-Step Demo

This notebook decomposes the same online retrieval path used by the Principia web app when the UI calls `POST /api/v1/research/start`.

The relevant web path is:

1. `principia/server.py` handles `/api/v1/research/start`.
2. The background worker calls `PrincipiaEngine.v1_research_project(...)`.
3. `v1_research_project(...)` delegates to `v2_research_project(...)`.
4. `v2_research_project(...)` prepares the search goal and calls `search_hybrid_sources(...)`.
5. `search_hybrid_sources(...)` builds a `RetrievalConfig` and calls `WorkRetriever.search(...)`.
6. `WorkRetriever.search(...)` plans queries, fans out to public metadata sources, deduplicates, ranks, optionally reranks with an LLM, and selects final works.

This notebook does not write project records to `data/principia.sqlite`. It only runs the online retrieval portion and exposes intermediate data structures.

## 1. Environment Setup

Run from this repository checkout. The setup cell puts the web app package and the shared retrieval package on `sys.path` so the notebook uses the local source code currently under development.

If `SILICONFLOW_API_KEY` or `OPENAI_API_KEY` is configured, the notebook follows the web behavior and enables LLM query planning plus LLM reranking. Without a key, it still uses the same online public metadata sources and deterministic ranking.

In [1]:
from __future__ import annotations

import json
import math
import sys
import tempfile
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

from IPython.display import Markdown, display


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / "principia").is_dir() and (path / "Principia-v1.3" / "src" / "principia_retrieval").is_dir():
            return path
        if path.name == "Principia-v1.3" and (path.parent / "principia").is_dir():
            return path.parent
    raise RuntimeError("Could not locate the Principia repository root.")


REPO_ROOT = find_repo_root()
WEB_APP_PATH = REPO_ROOT
RETRIEVAL_SRC_PATH = REPO_ROOT / "Principia-v1.3" / "src"

for path in [str(RETRIEVAL_SRC_PATH), str(WEB_APP_PATH)]:
    if path in sys.path:
        sys.path.remove(path)
sys.path.insert(0, str(RETRIEVAL_SRC_PATH))
sys.path.insert(0, str(WEB_APP_PATH))

from principia.engine import PrincipiaEngine
from principia.llm_client import LLMClient
from principia.research_sources import search_hybrid_sources
from principia.storage import Store
from principia_retrieval import RetrievalConfig, WorkRetriever
from principia_retrieval.planner import QueryPlanner
from principia_retrieval.ranking import deterministic_rank, final_select, has_exact_entity, llm_rerank
from principia_retrieval.sources import default_sources, fetch_source
from principia_retrieval.utils import llm_available, ordered_unique, strip_internal
from principia_retrieval.works import dedupe_works


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def markdown_table(rows: list[dict[str, Any]], columns: list[str], *, max_rows: int = 20) -> str:
    rows = rows[:max_rows]
    if not rows:
        return "_No rows._"
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join("---" for _ in columns) + " |"
    lines = [header, separator]
    for row in rows:
        values = []
        for column in columns:
            value = str(row.get(column, ""))
            value = value.replace("\n", " ").replace("|", "\\|")
            if len(value) > 120:
                value = value[:117] + "..."
            values.append(value)
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


def show_table(rows: list[dict[str, Any]], columns: list[str], *, max_rows: int = 20) -> None:
    display(Markdown(markdown_table(rows, columns, max_rows=max_rows)))


def work_source_name(work: dict[str, Any]) -> str:
    signals = work.get("community_signals") or work.get("metadata") or {}
    return str(signals.get("source") or work.get("source") or "unknown")


print(f"Repository root: {REPO_ROOT}")


Repository root: /home/supernova/Desktop/AI_projects/Principia


## 2. Web Request Parameters

These mirror the payload fields sent by the web UI to `/api/v1/research/start`. `TARGET_WORKS = 100` is the backend default. Lower it only when you want a faster exploratory run.

In [2]:
FIELD_ID = "notebook-web-retrieval-demo"
GOAL_TEXT = (
    "Please design an MAS framework where LLMs are interacting machine dialects like social interaction "
    "to improve the reasoning accuracy and reducing the tokens completion and cost."
)
MODEL_MODE = "auto"
TARGET_WORKS = 20
TIMEOUT_SECONDS = 12

request_payload = {
    "field_id": FIELD_ID,
    "goal_text": GOAL_TEXT,
    "model_mode": MODEL_MODE,
    "target_works": TARGET_WORKS,
    "force_refresh": True,
}

show_json(request_payload)


{
  "field_id": "notebook-web-retrieval-demo",
  "goal_text": "Please design an MAS framework where LLMs are interacting machine dialects like social interaction to improve the reasoning accuracy and reducing the tokens completion and cost.",
  "model_mode": "auto",
  "target_works": 20,
  "force_refresh": true
}


## 3. Engine Query Preparation

`v2_research_project(...)` performs two preparation steps before metadata search:

- `_v2_english_search_goal(...)`: translate or normalize the user's research goal for academic search.
- `_v2_research_query(...)`: add keyword terms and special expansions used for the progress-visible planned query.

`search_hybrid_sources(query, original_goal=search_goal)` then passes `original_goal` into `WorkRetriever.search(...)`, so the shared retriever plans from the English search goal when one exists.

In [3]:
llm = LLMClient()

# Use a temporary store so this notebook does not mutate the real web database.
with tempfile.TemporaryDirectory(prefix="principia-retrieval-demo-") as tmpdir:
    demo_store = Store(Path(tmpdir) / "data" / "principia.sqlite")
    engine = PrincipiaEngine(store=demo_store, llm=llm)
    search_goal = engine._v2_english_search_goal(GOAL_TEXT, model_mode=MODEL_MODE)
    planned_query = engine._v2_research_query(search_goal or GOAL_TEXT)
    model_meta = engine._v2_model_meta(MODEL_MODE)

retrieval_goal = search_goal or GOAL_TEXT
llm_for_retrieval = llm if llm.available() else None

show_json(
    {
        "llm_available": bool(llm_for_retrieval),
        "model_meta": model_meta,
        "original_goal": GOAL_TEXT,
        "search_goal": search_goal,
        "planned_query_for_progress": planned_query,
        "retriever_goal": retrieval_goal,
    }
)


{
  "llm_available": true,
  "model_meta": {
    "model_mode": "auto",
    "provider": "siliconflow",
    "model_name": "Qwen/Qwen3.6-27B"
  },
  "original_goal": "Please design an MAS framework where LLMs are interacting machine dialects like social interaction to improve the reasoning accuracy and reducing the tokens completion and cost.",
  "search_goal": "Please design an MAS framework where LLMs are interacting machine dialects like social interaction to improve the reasoning accuracy and reducing the tokens completion and cost.",
  "planned_query_for_progress": "Please design an MAS framework where LLMs are interacting machine dialects like social interaction to improve the reasoning accuracy and reducing the tokens completion and cost. accuracy completion cost dialects framework interacting interaction llms machine mas",
  "retriever_goal": "Please design an MAS framework where LLMs are interacting machine dialects like social interaction to improve the reasoning accuracy and re

## 4. Build The Same RetrievalConfig As `search_hybrid_sources`

This cell mirrors `principia/research_sources.py::search_hybrid_sources(...)`.

In [4]:
normalized_query = " ".join(str(planned_query or "").split())

config = RetrievalConfig(
    use_llm_planner=llm_for_retrieval is not None,
    use_llm_rerank=bool(llm_for_retrieval),
    max_raw_candidates=max(100, int(TARGET_WORKS or 100) * 4),
    max_queries=8,
)

show_json(
    {
        "normalized_query_nonempty": bool(normalized_query),
        "use_llm_planner": config.use_llm_planner,
        "use_llm_rerank": config.use_llm_rerank,
        "max_raw_candidates": config.max_raw_candidates,
        "min_relevance": config.min_relevance,
        "max_queries": config.max_queries,
        "llm_batch_size": config.llm_batch_size,
    }
)


{
  "normalized_query_nonempty": true,
  "use_llm_planner": true,
  "use_llm_rerank": true,
  "max_raw_candidates": 100,
  "min_relevance": 0.08,
  "max_queries": 8,
  "llm_batch_size": 24
}


## 5. Query Planning

`WorkRetriever.search(...)` creates a `QueryPlanner` and calls `plan(retriever_goal)`. With an available LLM, this includes LLM-generated search queries. Without an LLM, it uses deterministic planning only.

When LLM queries are available, the planner mixes query sources before source fan-out: a small number of deterministic anchor queries, a larger block of LLM-planned queries, and one final fallback query equal to the original goal text. This prevents deterministic queries from consuming the whole `max_queries` budget while preserving an exact-goal fallback.

The current deterministic planner is intentionally generic: it does not classify the goal as AI/non-AI, does not inject fixed AI/domain queries, and does not create goal-specific exclude terms. `ai_intent`, `domain_hints`, and `exclude_terms` are displayed below only as backward-compatible fields.

In [5]:
planner = QueryPlanner(llm_for_retrieval, use_llm=config.use_llm_planner, model_mode="auto")
plan = planner.plan(retrieval_goal)

source_registry = default_sources()
source_names = config.source_names or list(source_registry)
queries = ordered_unique(plan.search_queries or [retrieval_goal])[: max(1, config.max_queries)]
per_query = max(8, min(25, math.ceil(config.max_raw_candidates / max(1, len(source_names) * len(queries)))))
max_workers = min(8, max(1, len(source_names) * min(len(queries), 3)))

show_json(
    {
        "ai_intent": plan.ai_intent,
        "domain_hints": plan.domain_hints,
        "entities": plan.entities,
        "key_phrases": plan.key_phrases,
        "exclude_terms": plan.exclude_terms,
        "trace": plan.trace,
        "source_names": source_names,
        "queries_used": queries,
        "per_query_limit": per_query,
        "max_workers": max_workers,
    }
)


{
  "ai_intent": false,
  "domain_hints": [],
  "entities": [
    "MAS",
    "LLMs",
    "multi-agent systems",
    "large language models",
    "machine dialects",
    "social interaction",
    "reasoning accuracy",
    "token completion",
    "inference cost"
  ],
  "key_phrases": [
    "please design mas",
    "design mas framework",
    "mas framework where",
    "framework where llms",
    "where llms interacting",
    "llms interacting machine",
    "interacting machine dialect",
    "machine dialect like",
    "dialect like social",
    "like social interaction",
    "social interaction improve",
    "interaction improve reasoning",
    "improve reasoning accuracy",
    "reasoning accuracy reducing",
    "accuracy reducing token",
    "reducing token completion",
    "token completion cost",
    "please design",
    "design mas",
    "mas framework",
    "framework where",
    "where llms",
    "llms interacting",
    "interacting machine",
    "machine dialect",
    "dialect li

## 6. Online Source Fan-Out

This is the online metadata retrieval step. It calls the same source registry as the web app: arXiv, OpenAlex, Crossref, and Semantic Scholar.

`fetch_source(...)` normalizes each source result into the shared work schema and returns an empty list if a source request fails or is rate-limited.

In [6]:
raw: list[dict[str, Any]] = []
source_trace: list[dict[str, Any]] = []

start = time.time()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_meta = {}
    for name in source_names:
        source = source_registry.get(name)
        if not source:
            continue
        for query in queries:
            future = executor.submit(fetch_source, name, source, query, per_query, TIMEOUT_SECONDS)
            future_meta[future] = {"source": name, "query": query}

    for future in as_completed(future_meta):
        meta = future_meta[future]
        rows = future.result()
        raw.extend(rows)
        source_trace.append(
            {
                "source": meta["source"],
                "returned": len(rows),
                "query": meta["query"],
                "sample_titles": "; ".join(str(row.get("title") or "") for row in rows[:2]),
            }
        )
        if len(raw) >= config.max_raw_candidates * 2:
            break

elapsed = round(time.time() - start, 2)
source_trace.sort(key=lambda row: (row["source"], row["query"]))

source_counts = Counter(row["source"] for row in raw)
show_json({"elapsed_seconds": elapsed, "raw_count": len(raw), "source_counts": dict(source_counts)})
show_table(source_trace, ["source", "returned", "query", "sample_titles"], max_rows=40)

if not raw:
    print("No online source returned works. Check network access, API availability, or rate limits, then rerun this cell.")


{
  "elapsed_seconds": 11.55,
  "raw_count": 84,
  "source_counts": {
    "arxiv": 44,
    "semantic_scholar": 16,
    "crossref": 24
  }
}


| source | returned | query | sample_titles |
| --- | --- | --- | --- |
| arxiv | 2 | LLM agent communication protocols cost reduction | COALESCE: Economic and Security Dynamics of Skill-Based Task Outsourcing Among Team of Autonomous LLM Agents; MAS-ZER... |
| arxiv | 3 | LLM agents dialogue-based reasoning token efficiency | MemMachine: A Ground-Truth-Preserving Memory System for Personalized AI Agents; Mem0: Building Production-Ready AI Ag... |
| arxiv | 8 | design mas framework | Unified-MAS: Universally Generating Domain-Specific Nodes for Empowering Automatic Multi-Agent Systems; MAS-ZERO: Des... |
| arxiv | 8 | efficient reasoning multi-agent large language models | SMAGDi: Socratic Multi Agent Interaction Graph Distillation for Efficient High Accuracy Reasoning; ARCANE: A Multi-Ag... |
| arxiv | 2 | machine dialects LLM interaction reasoning improvement | Self-reflecting Large Language Models: A Hegelian Dialectical Approach; AgentSM: Semantic Memory for Agentic Text-to-SQL |
| arxiv | 8 | multi-agent systems LLM social interaction reasoning accuracy | MINDGAMES: A Live Arena for Evaluating Social and Strategic Reasoning in Multi-Agent LLMs; Conformity Dynamics in LLM... |
| arxiv | 5 | please design mas | Security Considerations for Multi-agent Systems; Future Circular Collider Feasibility Study Report: Volume 2, Acceler... |
| arxiv | 8 | reducing inference cost multi-agent LLM frameworks | Council Mode: A Heterogeneous Multi-Agent Consensus Framework for Reducing LLM Hallucination and Bias; MALBO: Optimiz... |
| crossref | 0 | LLM agent communication protocols cost reduction |  |
| crossref | 8 | LLM agents dialogue-based reasoning token efficiency | Token Cost Optimization in an LLM Agent for JSON-Based System Modeling from Dialogue History; No Future for LLM-based... |
| crossref | 0 | design mas framework |  |
| crossref | 0 | efficient reasoning multi-agent large language models |  |
| crossref | 8 | machine dialects LLM interaction reasoning improvement | ET-LLM: An External Pseudo-State Interaction Framework for Stabilizing Reasoning Stance in Stateless Large Language M... |
| crossref | 0 | multi-agent systems LLM social interaction reasoning accuracy |  |
| crossref | 8 | please design mas | The Evolution of Rotation Group Bias: Will the Real Unemployment Rate Please Stand Up?; The Evolution of Rotation Gro... |
| crossref | 0 | reducing inference cost multi-agent LLM frameworks |  |
| openalex | 0 | LLM agent communication protocols cost reduction |  |
| openalex | 0 | LLM agents dialogue-based reasoning token efficiency |  |
| openalex | 0 | design mas framework |  |
| openalex | 0 | efficient reasoning multi-agent large language models |  |
| openalex | 0 | machine dialects LLM interaction reasoning improvement |  |
| openalex | 0 | multi-agent systems LLM social interaction reasoning accuracy |  |
| openalex | 0 | please design mas |  |
| openalex | 0 | reducing inference cost multi-agent LLM frameworks |  |
| semantic_scholar | 0 | LLM agent communication protocols cost reduction |  |
| semantic_scholar | 0 | LLM agents dialogue-based reasoning token efficiency |  |
| semantic_scholar | 8 | design mas framework | MAS-Shield: A Defense Framework for Secure and Efficient LLM MAS; MAS-ISP: A Proxy-Free Online Hyperparameter Optimiz... |
| semantic_scholar | 0 | efficient reasoning multi-agent large language models |  |
| semantic_scholar | 0 | machine dialects LLM interaction reasoning improvement |  |
| semantic_scholar | 8 | multi-agent systems LLM social interaction reasoning accuracy | Which Agent Causes Task Failures and When? On Automated Failure Attribution of LLM Multi-Agent Systems; RCR-Router: E... |
| semantic_scholar | 0 | please design mas |  |
| semantic_scholar | 0 | reducing inference cost multi-agent LLM frameworks |  |

## 7. Normalize And Deduplicate Works

`fetch_source(...)` already normalizes individual source rows. `dedupe_works(...)` merges duplicates using DOI, arXiv id, OpenAlex id, Semantic Scholar id, and normalized title keys.

In [7]:
candidates = dedupe_works(raw)

dedupe_summary = {
    "raw_count": len(raw),
    "deduped_candidate_count": len(candidates),
    "removed_as_duplicates": max(0, len(raw) - len(candidates)),
    "candidate_sources": dict(Counter(work_source_name(work) for work in candidates)),
}
show_json(dedupe_summary)

candidate_rows = [
    {
        "index": index + 1,
        "source": work_source_name(work),
        "year": work.get("year"),
        "citations": work.get("citation_count"),
        "title": work.get("title"),
        "venue": work.get("venue_or_source"),
    }
    for index, work in enumerate(candidates[:30])
]
show_table(candidate_rows, ["index", "source", "year", "citations", "title", "venue"], max_rows=30)


{
  "raw_count": 84,
  "deduped_candidate_count": 78,
  "removed_as_duplicates": 6,
  "candidate_sources": {
    "arxiv": 39,
    "semantic_scholar": 16,
    "crossref": 23
  }
}


| index | source | year | citations | title | venue |
| --- | --- | --- | --- | --- | --- |
| 1 | arxiv | 2025 | None | Self-reflecting Large Language Models: A Hegelian Dialectical Approach | arXiv |
| 2 | arxiv | 2026 | None | AgentSM: Semantic Memory for Agentic Text-to-SQL | arXiv |
| 3 | arxiv | 2025 | None | COALESCE: Economic and Security Dynamics of Skill-Based Task Outsourcing Among Team of Autonomous LLM Agents | arXiv |
| 4 | semantic_scholar | 2025 | 20 | MAS-ZERO: Designing Multi-Agent Systems with Zero Supervision | arXiv.org |
| 5 | arxiv | 2026 | None | MemMachine: A Ground-Truth-Preserving Memory System for Personalized AI Agents | arXiv |
| 6 | arxiv | 2025 | None | Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory | arXiv |
| 7 | arxiv | 2025 | None | StreamVLN: Streaming Vision-and-Language Navigation via SlowFast Context Modeling | arXiv |
| 8 | arxiv | 2026 | None | Council Mode: A Heterogeneous Multi-Agent Consensus Framework for Reducing LLM Hallucination and Bias | arXiv |
| 9 | arxiv | 2025 | None | MALBO: Optimizing LLM-Based Multi-Agent Teams via Multi-Objective Bayesian Optimization | arXiv |
| 10 | arxiv | 2026 | None | GAMMAF: A Common Framework for Graph-Based Anomaly Monitoring Benchmarking in LLM Multi-Agent Systems | arXiv |
| 11 | arxiv | 2025 | None | Collaborative Belief Reasoning with LLMs for Efficient Multi-Agent Collaboration | arXiv |
| 12 | arxiv | 2026 | None | SC-MAS: Constructing Cost-Efficient Multi-Agent Systems with Edge-Level Heterogeneous Collaboration | arXiv |
| 13 | arxiv | 2025 | None | Scaling Graph Chain-of-Thought Reasoning: A Multi-Agent Framework with Efficient LLM Serving | arXiv |
| 14 | arxiv | 2025 | None | iMAD: Intelligent Multi-Agent Debate for Efficient and Accurate LLM Inference | arXiv |
| 15 | arxiv | 2026 | None | Learning Transferable Topology Priors for Multi-Agent LLM Collaboration Across Domains | arXiv |
| 16 | arxiv | 2026 | None | MINDGAMES: A Live Arena for Evaluating Social and Strategic Reasoning in Multi-Agent LLMs | arXiv |
| 17 | semantic_scholar | 2026 | 4 | Conformity Dynamics in LLM Multi-Agent Systems: The Roles of Topology and Self-Social Weighting | arXiv.org |
| 18 | arxiv | 2026 | None | WhatIf: Interactive Exploration of LLM-Powered Social Simulations for Policy Reasoning | arXiv |
| 19 | arxiv | 2026 | None | Beyond Self-Interest: Modeling Social-Oriented Motivation for Human-like Multi-Agent Interactions | arXiv |
| 20 | arxiv | 2026 | None | Game-Theoretic Lens on LLM-based Multi-Agent Systems | arXiv |
| 21 | arxiv | 2026 | None | Topology-Aware LLM-Driven Social Simulation: A Unified Framework for Efficient and Realistic Agent Dynamics | arXiv |
| 22 | arxiv | 2026 | None | The Bystander Effect in Multi-Agent Reasoning: Quantifying Cognitive Loafing in Collaborative Interactions | arXiv |
| 23 | arxiv | 2025 | None | Towards Simulating Social Influence Dynamics with LLM-based Multi-agents | arXiv |
| 24 | arxiv | 2025 | None | SMAGDi: Socratic Multi Agent Interaction Graph Distillation for Efficient High Accuracy Reasoning | arXiv |
| 25 | arxiv | 2025 | None | ARCANE: A Multi-Agent Framework for Interpretable and Configurable Alignment | arXiv |
| 26 | arxiv | 2024 | None | KG-Agent: An Efficient Autonomous Agent Framework for Complex Reasoning over Knowledge Graph | arXiv |
| 27 | arxiv | 2026 | None | Fanar-Sadiq: A Multi-Agent Architecture for Grounded Islamic QA | arXiv |
| 28 | arxiv | 2024 | None | MAGDi: Structured Distillation of Multi-Agent Interaction Graphs Improves Reasoning in Smaller Language Models | arXiv |
| 29 | arxiv | 2024 | None | Multi-Agent Large Language Models for Conversational Task-Solving | arXiv |
| 30 | arxiv | 2025 | None | Be My Eyes: Extending Large Language Models to New Modalities Through Multi-Agent Collaboration | arXiv |

## 8. Deterministic Ranking And Prefilter

`deterministic_rank(...)` scores candidates using semantic-lite overlap, metadata quality, and exact entity bonuses. The prefilter keeps works above `config.min_relevance` or works with exact entity matches. If nothing passes, the web code falls back to the top deterministic candidates.

In [8]:
scored = deterministic_rank(retrieval_goal, candidates, plan)
prefiltered = [item for item in scored if item["_retrieval_score"] >= config.min_relevance or has_exact_entity(item, plan)]
if not prefiltered:
    prefiltered = scored[: max(TARGET_WORKS, 20)]

show_json(
    {
        "scored_count": len(scored),
        "prefiltered_count": len(prefiltered),
        "min_relevance": config.min_relevance,
    }
)

rank_rows = [
    {
        "rank": index + 1,
        "score": round(float(work.get("_retrieval_score", 0.0)), 4),
        "relation": work.get("relation_label"),
        "source": work_source_name(work),
        "year": work.get("year"),
        "title": work.get("title"),
        "rationale": work.get("retrieval_rationale"),
    }
    for index, work in enumerate(scored[:30])
]
show_table(rank_rows, ["rank", "score", "relation", "source", "year", "title", "rationale"], max_rows=30)


{
  "scored_count": 78,
  "prefiltered_count": 75,
  "min_relevance": 0.08
}


| rank | score | relation | source | year | title | rationale |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 1.3928 | direct | arxiv | 2026 | SC-MAS: Constructing Cost-Efficient Multi-Agent Systems with Edge-Level Heterogeneous Collaboration | matches target entity; matches mas, framework, where |
| 2 | 1.3248 | direct | semantic_scholar | 2026 | Conformity Dynamics in LLM Multi-Agent Systems: The Roles of Topology and Self-Social Weighting | matches target entity; matches social interaction, mas, where |
| 3 | 1.3165 | direct | arxiv | 2026 | Game-Theoretic Lens on LLM-based Multi-Agent Systems | matches target entity; matches design, mas, framework |
| 4 | 1.3123 | direct | arxiv | 2025 | Collaborative Belief Reasoning with LLMs for Efficient Multi-Agent Collaboration | matches target entity; matches framework, llms, like |
| 5 | 1.3046 | direct | semantic_scholar | 2025 | MAS-ZERO: Designing Multi-Agent Systems with Zero Supervision | matches target entity; matches design mas, design, mas |
| 6 | 1.2997 | direct | arxiv | 2024 | Multi-Agent Large Language Models for Conversational Task-Solving | matches target entity; matches framework, where, llms |
| 7 | 1.2869 | direct | semantic_scholar | 2025 | Simulating Social Behavior of LLM-Based Autonomous Negotiator Agents in a Game-Theoretical Framework Using Multi-Agen... | matches target entity; matches design, framework, llms |
| 8 | 1.2826 | direct | arxiv | 2026 | Council Mode: A Heterogeneous Multi-Agent Consensus Framework for Reducing LLM Hallucination and Bias | matches target entity; matches framework, where, llms |
| 9 | 1.2826 | direct | arxiv | 2026 | Unified-MAS: Universally Generating Domain-Specific Nodes for Empowering Automatic Multi-Agent Systems | matches target entity; matches design, mas, framework |
| 10 | 1.2737 | direct | semantic_scholar | 2026 | SocialGrid: A Benchmark for Planning and Social Reasoning in Embodied Multi-Agent Systems | matches target entity; matches llms, social |
| 11 | 1.2615 | direct | arxiv | 2025 | SMAGDi: Socratic Multi Agent Interaction Graph Distillation for Efficient High Accuracy Reasoning | matches target entity; matches reasoning accuracy, mas, framework |
| 12 | 1.2318 | direct | arxiv | 2026 | GAMMAF: A Common Framework for Graph-Based Anomaly Monitoring Benchmarking in LLM Multi-Agent Systems | matches target entity; matches design, mas, framework |
| 13 | 1.2119 | direct | semantic_scholar | 2025 | ReSo: A Reward-driven Self-organizing LLM-based Multi-Agent System for Reasoning Tasks | matches target entity; matches mas framework, mas, framework |
| 14 | 1.1978 | direct | arxiv | 2025 | Scaling Graph Chain-of-Thought Reasoning: A Multi-Agent Framework with Efficient LLM Serving | matches target entity; matches design, framework, llms |
| 15 | 1.1975 | direct | semantic_scholar | 2025 | A Design Framework for Scalable and Adaptive Multi-Agent Coordination in Dynamic Environments: Addressing Concurrent ... | matches target entity; matches design, mas, framework |
| 16 | 1.1937 | direct | arxiv | 2025 | StreamVLN: Streaming Vision-and-Language Navigation via SlowFast Context Modeling | matches target entity; matches design, framework, llms |
| 17 | 1.1767 | direct | arxiv | 2025 | MALBO: Optimizing LLM-Based Multi-Agent Teams via Multi-Objective Bayesian Optimization | matches target entity; matches design, framework, llms |
| 18 | 1.1765 | direct | arxiv | 2026 | Topology-Aware LLM-Driven Social Simulation: A Unified Framework for Efficient and Realistic Agent Dynamics | matches target entity; matches reducing token, framework, llms |
| 19 | 1.1641 | direct | crossref | 2026 | Teacher–Student–Machine Interaction Autonomous Learning: A Structured LLM-Integrated Framework for Developing Indepen... | matches target entity; matches framework, llms, machine |
| 20 | 1.1258 | direct | arxiv | 2024 | KG-Agent: An Efficient Autonomous Agent Framework for Complex Reasoning over Knowledge Graph | matches target entity; matches design, framework, llms |
| 21 | 1.1254 | direct | semantic_scholar | 2026 | Seeing the Whole Elephant: A Benchmark for Failure Attribution in LLM-based Multi-Agent Systems | matches target entity; matches design, mas, where |
| 22 | 1.1087 | direct | arxiv | 2026 | The Bystander Effect in Multi-Agent Reasoning: Quantifying Cognitive Loafing in Collaborative Interactions | matches target entity; matches mas, where, social |
| 23 | 1.1087 | direct | arxiv | 2026 | Agentic Social Affordance Framework (ASAF): Agent Identity Design as a Collaboration Interface in Multi-Agent Systems | matches target entity; matches design, mas, framework |
| 24 | 1.1067 | direct | arxiv | 2026 | AgentSM: Semantic Memory for Agentic Text-to-SQL | matches target entity; matches design, mas, framework |
| 25 | 1.0998 | direct | crossref | 2026 | ET-LLM: An External Pseudo-State Interaction Framework for Stabilizing Reasoning Stance in Stateless Large Language M... | matches target entity; matches design, framework, llms |
| 26 | 1.0957 | direct | semantic_scholar | 2025 | Can Lessons From Human Teams Be Applied to Multi-Agent Systems? The Role of Structure, Diversity, and Interaction Dyn... | matches target entity; matches mas, framework, social |
| 27 | 1.0622 | direct | arxiv | 2025 | Adaptive Coopetition: Leveraging Coarse Verifier Signals for Resilient Multi-Agent LLM Reasoning | matches target entity; matches improve reasoning, framework, llms |
| 28 | 1.0579 | direct | arxiv | 2026 | Beyond Self-Interest: Modeling Social-Oriented Motivation for Human-like Multi-Agent Interactions | matches target entity; matches where, llms, like |
| 29 | 1.0537 | direct | semantic_scholar | 2025 | MAS-Shield: A Defense Framework for Secure and Efficient LLM MAS | matches target entity; matches design, mas, framework |
| 30 | 1.0367 | direct | arxiv | 2025 | Towards Simulating Social Influence Dynamics with LLM-based Multi-agents | matches target entity; matches social interaction, framework, where |

## 9. Optional LLM Rerank

This follows the web condition: rerank is enabled only when `config.use_llm_rerank` is true and the LLM client is available. For `TARGET_WORKS = 100`, the reranker may evaluate up to `max(TARGET_WORKS * 2, 50)` candidates in batches of `config.llm_batch_size`.

If you want to inspect only deterministic retrieval, set `FORCE_SKIP_LLM_RERANK = True` before running this cell.

In [9]:
FORCE_SKIP_LLM_RERANK = False

rerank_enabled = config.use_llm_rerank and llm_available(llm_for_retrieval) and not FORCE_SKIP_LLM_RERANK
rerank_input = prefiltered[: max(TARGET_WORKS * 2, 50)]

if rerank_enabled and rerank_input:
    reranked = llm_rerank(retrieval_goal, rerank_input, plan, llm_for_retrieval, batch_size=config.llm_batch_size)
    rerank_status = "llm_rerank_ran"
else:
    reranked = prefiltered
    rerank_status = "llm_rerank_skipped"

show_json(
    {
        "rerank_status": rerank_status,
        "rerank_input_count": len(rerank_input),
        "reranked_count": len(reranked),
        "llm_available": llm_available(llm_for_retrieval),
    }
)

rerank_rows = [
    {
        "rank": index + 1,
        "score": round(float(work.get("_retrieval_score", 0.0)), 4),
        "relation": work.get("relation_label"),
        "source": work_source_name(work),
        "year": work.get("year"),
        "title": work.get("title"),
        "rationale": work.get("retrieval_rationale"),
        "reject_reason": work.get("reject_reason"),
    }
    for index, work in enumerate(reranked[:30])
]
show_table(rerank_rows, ["rank", "score", "relation", "source", "year", "title", "rationale", "reject_reason"], max_rows=30)


{
  "rerank_status": "llm_rerank_ran",
  "rerank_input_count": 50,
  "reranked_count": 50,
  "llm_available": true
}


| rank | score | relation | source | year | title | rationale | reject_reason |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 1.2775 | direct | semantic_scholar | 2025 | RCR-Router: Efficient Role-Aware Context Routing for Multi-Agent LLM Systems with Structured Memory | Directly addresses token reduction and reasoning in MAS via role-aware context routing. |  |
| 2 | 1.2642 | direct | arxiv | 2026 | SC-MAS: Constructing Cost-Efficient Multi-Agent Systems with Edge-Level Heterogeneous Collaboration | Proposes SC-MAS framework using social capital theory for heterogeneous collaboration, explicitly targeting cost-effi... |  |
| 3 | 1.2267 | direct | arxiv | 2026 | Learning Transferable Topology Priors for Multi-Agent LLM Collaboration Across Domains | Reduces token consumption and improves reasoning efficiency via transferable topology priors in MAS. |  |
| 4 | 1.1725 | direct | arxiv | 2025 | SMAGDi: Socratic Multi Agent Interaction Graph Distillation for Efficient High Accuracy Reasoning | Introduces SMAGDi to distill multi-agent debate dynamics into efficient models, directly addressing reasoning accurac... |  |
| 5 | 1.1683 | direct | arxiv | 2024 | MAGDi: Structured Distillation of Multi-Agent Interaction Graphs Improves Reasoning in Smaller Language Models | Distills multi-agent interactions to reduce cost while maintaining reasoning accuracy in smaller models. |  |
| 6 | 1.1641 | direct | crossref | 2026 | Teacher–Student–Machine Interaction Autonomous Learning: A Structured LLM-Integrated Framework for Developing Indepen... | matches target entity; matches framework, llms, machine |  |
| 7 | 1.1559 | direct | semantic_scholar | 2026 | Conformity Dynamics in LLM Multi-Agent Systems: The Roles of Topology and Self-Social Weighting | Studies conformity dynamics and social interaction mechanisms in LLM MAS, providing insights into how social weightin... |  |
| 8 | 1.135 | direct | arxiv | 2025 | Adaptive Coopetition: Leveraging Coarse Verifier Signals for Resilient Multi-Agent LLM Reasoning | Improves reasoning accuracy via adaptive coopetition, addressing coordination efficiency in MAS. |  |
| 9 | 1.0975 | direct | arxiv | 2025 | Self-reflecting Large Language Models: A Hegelian Dialectical Approach | Uses dialectical interaction to improve reasoning, aligning with the social interaction goal. |  |
| 10 | 1.0537 | direct | semantic_scholar | 2025 | MAS-Shield: A Defense Framework for Secure and Efficient LLM MAS | matches target entity; matches design, mas, framework |  |
| 11 | 1.0266 | background | semantic_scholar | 2025 | Can Lessons From Human Teams Be Applied to Multi-Agent Systems? The Role of Structure, Diversity, and Interaction Dyn... | Explores team dynamics and interaction structures in MAS, providing context for social interaction design. |  |
| 12 | 1.01 | methodological | arxiv | 2025 | MALBO: Optimizing LLM-Based Multi-Agent Teams via Multi-Objective Bayesian Optimization | Provides MALBO framework for optimizing LLM agent teams via multi-objective Bayesian optimization, balancing accuracy... |  |
| 13 | 0.9475 | background | arxiv | 2025 | Collaborative Belief Reasoning with LLMs for Efficient Multi-Agent Collaboration | Proposes CoBel-World for collaborative belief reasoning, reducing redundant communication and improving efficiency in... |  |
| 14 | 0.9225 | background | arxiv | 2025 | Towards Simulating Social Influence Dynamics with LLM-based Multi-agents | Simulates social influence dynamics, offering insights into social interaction mechanisms. |  |
| 15 | 0.9223 | direct | arxiv | 2026 | Fanar-Sadiq: A Multi-Agent Architecture for Grounded Islamic QA | matches target entity; matches where, llms |  |
| 16 | 0.9057 | methodological | crossref | 2026 | ET-LLM: An External Pseudo-State Interaction Framework for Stabilizing Reasoning Stance in Stateless Large Language M... | Provides a framework for stabilizing reasoning stance, useful for consistent agent behavior. |  |
| 17 | 0.8892 | background | arxiv | 2026 | Topology-Aware LLM-Driven Social Simulation: A Unified Framework for Efficient and Realistic Agent Dynamics | Introduces TopoSim for topology-aware social simulation, offering context on how structural signals shape efficient a... |  |
| 18 | 0.8435 | methodological | semantic_scholar | 2026 | OFA-MAS: One-for-All Multi-Agent System Topology Design based on Mixture-of-Experts Graph Generative Models | Generates adaptive collaboration topologies, relevant to structuring efficient agent interactions. |  |
| 19 | 0.8267 | background | arxiv | 2026 | The Bystander Effect in Multi-Agent Reasoning: Quantifying Cognitive Loafing in Collaborative Interactions | Analyzes the 'Bystander Effect' and cognitive loafing in MAS, providing critical background on social pressure impact... |  |
| 20 | 0.7642 | background | arxiv | 2026 | Agentic Social Affordance Framework (ASAF): Agent Identity Design as a Collaboration Interface in Multi-Agent Systems | Proposes ASAF framework for agent identity design, relevant to structuring social interactions but less focused on co... |  |
| 21 | 0.76 | methodological | arxiv | 2025 | Be My Eyes: Extending Large Language Models to New Modalities Through Multi-Agent Collaboration | Demonstrates efficient multi-agent collaboration for reasoning, though focused on multimodality. |  |
| 22 | 0.7017 | background | arxiv | 2026 | Game-Theoretic Lens on LLM-based Multi-Agent Systems | Survey of LLM MAS through game-theoretic lens, providing theoretical background on strategic behaviors and social dyn... |  |
| 23 | 0.6392 | background | arxiv | 2026 | MemMachine: A Ground-Truth-Preserving Memory System for Personalized AI Agents | Addresses LLM agent memory and efficiency, but lacks multi-agent interaction focus. |  |
| 24 | 0.635 | background | arxiv | 2025 | ARCANE: A Multi-Agent Framework for Interpretable and Configurable Alignment | Focuses on alignment via multi-agent collaboration, less relevant to reasoning/cost goals. |  |
| 25 | 0.6308 | background | arxiv | 2024 | Multi-Agent Large Language Models for Conversational Task-Solving | Evaluates multi-agent conversational paradigms, offering general context on discussion structures but lacking specifi... |  |
| 26 | 0.5767 | background | arxiv | 2026 | Council Mode: A Heterogeneous Multi-Agent Consensus Framework for Reducing LLM Hallucination and Bias | Council Mode reduces hallucination via consensus, relevant to accuracy but less focused on token cost or social inter... |  |
| 27 | 0.5767 | background | arxiv | 2026 | MINDGAMES: A Live Arena for Evaluating Social and Strategic Reasoning in Multi-Agent LLMs | Evaluates social reasoning in MAS, providing benchmark context but not a framework. |  |
| 28 | 0.5282 | methodological | semantic_scholar | 2025 | ReSo: A Reward-driven Self-organizing LLM-based Multi-Agent System for Reasoning Tasks | ReSo uses reward-driven self-organization for reasoning tasks, offering methodological insights into optimizing MAS c... |  |
| 29 | 0.51 | methodological | arxiv | 2025 | Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory | Addresses memory efficiency, indirectly related to token reduction in long interactions. |  |
| 30 | 0.4701 | background | semantic_scholar | 2025 | Which Agent Causes Task Failures and When? On Automated Failure Attribution of LLM Multi-Agent Systems | Focuses on failure attribution, not directly on reasoning improvement or cost reduction. |  |

## 10. Final Selection And Ranking Trace

`final_select(...)` is the last selection step inside `WorkRetriever.search(...)`. The selected works are stripped of internal fields before they are returned to `search_hybrid_sources(...)` and then persisted by the engine.

In [10]:
selected = final_select(reranked, TARGET_WORKS, plan)
ranking_trace = [
    {
        "work_id": item.get("work_id", ""),
        "title": item.get("title", ""),
        "score": round(float(item.get("_retrieval_score", 0.0)), 4),
        "relation_label": item.get("relation_label", ""),
        "rationale": item.get("retrieval_rationale", ""),
        "reject_reason": item.get("reject_reason", ""),
    }
    for item in selected
]
public_selected_works = [strip_internal(item) for item in selected]

show_json({"selected_count": len(public_selected_works), "trace_count": len(ranking_trace)})

selected_rows = [
    {
        "rank": index + 1,
        "score": trace["score"],
        "relation": trace["relation_label"],
        "source": work_source_name(work),
        "year": work.get("year"),
        "title": work.get("title"),
        "url_or_doi": work.get("url_or_doi"),
    }
    for index, (work, trace) in enumerate(zip(public_selected_works, ranking_trace))
]
show_table(selected_rows, ["rank", "score", "relation", "source", "year", "title", "url_or_doi"], max_rows=TARGET_WORKS)


{
  "selected_count": 20,
  "trace_count": 20
}


| rank | score | relation | source | year | title | url_or_doi |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 1.2775 | direct | semantic_scholar | 2025 | RCR-Router: Efficient Role-Aware Context Routing for Multi-Agent LLM Systems with Structured Memory | https://www.semanticscholar.org/paper/25d2c50c05e9109f56b275f13948b0965edd103b |
| 2 | 1.2642 | direct | arxiv | 2026 | SC-MAS: Constructing Cost-Efficient Multi-Agent Systems with Edge-Level Heterogeneous Collaboration | http://arxiv.org/abs/2601.09434v1 |
| 3 | 1.2267 | direct | arxiv | 2026 | Learning Transferable Topology Priors for Multi-Agent LLM Collaboration Across Domains | http://arxiv.org/abs/2605.17359v1 |
| 4 | 1.1725 | direct | arxiv | 2025 | SMAGDi: Socratic Multi Agent Interaction Graph Distillation for Efficient High Accuracy Reasoning | http://arxiv.org/abs/2511.05528v1 |
| 5 | 1.1683 | direct | arxiv | 2024 | MAGDi: Structured Distillation of Multi-Agent Interaction Graphs Improves Reasoning in Smaller Language Models | http://arxiv.org/abs/2402.01620v2 |
| 6 | 1.1641 | direct | crossref | 2026 | Teacher–Student–Machine Interaction Autonomous Learning: A Structured LLM-Integrated Framework for Developing Indepen... | https://doi.org/10.21203/rs.3.rs-8818923/v1 |
| 7 | 1.1559 | direct | semantic_scholar | 2026 | Conformity Dynamics in LLM Multi-Agent Systems: The Roles of Topology and Self-Social Weighting | https://www.semanticscholar.org/paper/b68ef48f4af68bfa60805de9984a8ac05c0c2315 |
| 8 | 1.135 | direct | arxiv | 2025 | Adaptive Coopetition: Leveraging Coarse Verifier Signals for Resilient Multi-Agent LLM Reasoning | http://arxiv.org/abs/2510.18179v2 |
| 9 | 1.0975 | direct | arxiv | 2025 | Self-reflecting Large Language Models: A Hegelian Dialectical Approach | http://arxiv.org/abs/2501.14917v6 |
| 10 | 1.0537 | direct | semantic_scholar | 2025 | MAS-Shield: A Defense Framework for Secure and Efficient LLM MAS | https://www.semanticscholar.org/paper/9c2f157eb7c4c08b5a3a61673881461729ea5b15 |
| 11 | 1.0266 | background | semantic_scholar | 2025 | Can Lessons From Human Teams Be Applied to Multi-Agent Systems? The Role of Structure, Diversity, and Interaction Dyn... | https://www.semanticscholar.org/paper/627ab0139699427ddedd0d9abd17b7834fdf320f |
| 12 | 1.01 | methodological | arxiv | 2025 | MALBO: Optimizing LLM-Based Multi-Agent Teams via Multi-Objective Bayesian Optimization | http://arxiv.org/abs/2511.11788v1 |
| 13 | 0.9475 | background | arxiv | 2025 | Collaborative Belief Reasoning with LLMs for Efficient Multi-Agent Collaboration | http://arxiv.org/abs/2509.21981v3 |
| 14 | 0.9225 | background | arxiv | 2025 | Towards Simulating Social Influence Dynamics with LLM-based Multi-agents | http://arxiv.org/abs/2507.22467v1 |
| 15 | 0.9223 | direct | arxiv | 2026 | Fanar-Sadiq: A Multi-Agent Architecture for Grounded Islamic QA | http://arxiv.org/abs/2603.08501v3 |
| 16 | 0.9057 | methodological | crossref | 2026 | ET-LLM: An External Pseudo-State Interaction Framework for Stabilizing Reasoning Stance in Stateless Large Language M... | https://doi.org/10.2139/ssrn.6096766 |
| 17 | 0.8892 | background | arxiv | 2026 | Topology-Aware LLM-Driven Social Simulation: A Unified Framework for Efficient and Realistic Agent Dynamics | http://arxiv.org/abs/2604.18011v2 |
| 18 | 0.8435 | methodological | semantic_scholar | 2026 | OFA-MAS: One-for-All Multi-Agent System Topology Design based on Mixture-of-Experts Graph Generative Models | https://www.semanticscholar.org/paper/27dbd36d88fdc3e6f178de7534b4243f32ad2a71 |
| 19 | 0.8267 | background | arxiv | 2026 | The Bystander Effect in Multi-Agent Reasoning: Quantifying Cognitive Loafing in Collaborative Interactions | http://arxiv.org/abs/2605.10698v1 |
| 20 | 0.7642 | background | arxiv | 2026 | Agentic Social Affordance Framework (ASAF): Agent Identity Design as a Collaboration Interface in Multi-Agent Systems | http://arxiv.org/abs/2606.09832v2 |

## 11. Optional One-Call Web Wrapper Check

The cells above expanded the internals of `search_hybrid_sources(...)`. Set `RUN_WRAPPER_CHECK = True` to run the real wrapper once as a cross-check. This performs another online retrieval, so it is disabled by default.

In [11]:
RUN_WRAPPER_CHECK = False

if RUN_WRAPPER_CHECK:
    wrapper_works = search_hybrid_sources(
        normalized_query,
        max_results=TARGET_WORKS,
        timeout=TIMEOUT_SECONDS,
        llm=llm_for_retrieval,
        original_goal=retrieval_goal,
    )
    wrapper_rows = [
        {
            "rank": index + 1,
            "source": work_source_name(work),
            "year": work.get("year"),
            "title": work.get("title"),
            "url_or_doi": work.get("url_or_doi"),
        }
        for index, work in enumerate(wrapper_works)
    ]
    show_json({"wrapper_selected_count": len(wrapper_works)})
    show_table(wrapper_rows, ["rank", "source", "year", "title", "url_or_doi"], max_rows=TARGET_WORKS)
else:
    print("Wrapper check is disabled. Set RUN_WRAPPER_CHECK = True to run search_hybrid_sources(...) end to end.")


Wrapper check is disabled. Set RUN_WRAPPER_CHECK = True to run search_hybrid_sources(...) end to end.
